In [ ]:
import base64, os, pathlib, subprocess
from google.colab import userdata

REPO_URL = "https://github.com/devlucascfarias/logos-3.git"
REPO_BRANCH = "main"
WORKDIR = "/content/logos-3"
repo = pathlib.Path(WORKDIR)

token = userdata.get("GH_TOKEN")
git = ["git"]
if token:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git += ["-c", f"http.extraHeader=Authorization: Basic {auth}"]

if (repo / ".git").exists():
    subprocess.run(git + ["-C", WORKDIR, "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
elif repo.exists():
    raise RuntimeError(f"{WORKDIR} existe, mas não é um repositório Git")
else:
    subprocess.run(git + ["clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, WORKDIR], check=True)

token = auth = None
os.chdir(WORKDIR)
print(f"Repositório sincronizado em {os.getcwd()}")

# Qwen3-8B QLoRA em NVIDIA L4

Pipeline da receita SFT. Comece com `SMOKE_TEST=True`; o treino completo só deve ser iniciado depois que dados, testes e ambiente passarem.

In [ ]:
SMOKE_TEST = True
STAGE = "baseline"  # baseline, main ou agentic
RUN_TRAINING = True
FRESH_SMOKE_RUN = True  # arquiva o adapter anterior antes de retreinar

TOKEN_BUDGET = 250_000 if SMOKE_TEST else None
MAX_SOURCE_ROWS = 5_000 if SMOKE_TEST else None
# None preserva num_train_epochs=1; evita várias passagens pela amostra smoke.
MAX_STEPS = None
MAX_TRAIN_SAMPLES = 250 if SMOKE_TEST else None
print({"stage": STAGE, "smoke": SMOKE_TEST, "token_budget": TOKEN_BUDGET})

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token, add_to_git_credential=False)
else:
    print("HF_TOKEN não definido; apenas fontes públicas sem aceite funcionarão.")
hf_token = None

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/environment_check.py"], check=True)

In [ ]:
command = [sys.executable, "scripts/prepare_data.py", "--stage", STAGE]
if TOKEN_BUDGET is not None:
    command += ["--token-budget", str(TOKEN_BUDGET)]
if MAX_SOURCE_ROWS is not None:
    command += ["--max-source-rows", str(MAX_SOURCE_ROWS)]
subprocess.run(command, check=True)

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

In [ ]:
import codecs, datetime, os, pathlib, shutil, subprocess, sys

if RUN_TRAINING:
    fresh_run = SMOKE_TEST and FRESH_SMOKE_RUN
    if fresh_run:
        timestamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        archive_root = pathlib.Path(WORKDIR) / "outputs" / "archive" / f"{STAGE}-{timestamp}"
        old_outputs = {
            "checkpoints": pathlib.Path(WORKDIR) / "outputs" / "checkpoints" / STAGE,
            "adapter": pathlib.Path(WORKDIR) / "outputs" / "adapters" / STAGE,
        }
        for label, source in old_outputs.items():
            if source.exists():
                archive_root.mkdir(parents=True, exist_ok=True)
                shutil.move(str(source), str(archive_root / label))
                print(f"Arquivado: {source} -> {archive_root / label}")
        FRESH_SMOKE_RUN = False
    command = [sys.executable, "-u", "scripts/train_sft.py", "--stage", STAGE]
    if not fresh_run:
        command += ["--resume-from-checkpoint", "auto"]
    if MAX_STEPS is not None:
        command += ["--max-steps", str(MAX_STEPS)]
    if MAX_TRAIN_SAMPLES is not None:
        command += ["--max-train-samples", str(MAX_TRAIN_SAMPLES)]
    print("Iniciando treino com barra de progresso e ETA...", flush=True)
    log_path = os.path.join(WORKDIR, "outputs", "logs", f"{STAGE}_train.log")
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    process = subprocess.Popen(
        command,
        cwd=WORKDIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    assert process.stdout is not None
    with open(log_path, "wb") as log_file:
        while True:
            chunk = os.read(process.stdout.fileno(), 4096)
            if not chunk:
                break
            log_file.write(chunk)
            log_file.flush()
            sys.stdout.write(decoder.decode(chunk))
            sys.stdout.flush()
        sys.stdout.write(decoder.decode(b"", final=True))
        sys.stdout.flush()
    return_code = process.wait()
    print(f"\nLog salvo em: {log_path}", flush=True)
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
else:
    print("RUN_TRAINING=False: dados e testes prontos; treino não iniciado.")

## Comparação cega: modelo-base vs. adapter baseline

Esta etapa gera respostas para os mesmos prompts inéditos, com geração determinística. As identidades são embaralhadas como A/B; avalie `comparison.md` antes de abrir `mapping.json`.

In [ ]:
import codecs, os, subprocess, sys

EVAL_OUTPUT_DIR = f"outputs/evaluations/{STAGE}_smoke"
command = [
    sys.executable, "-u", "scripts/compare_adapter.py",
    "--stage", STAGE,
    "--output-dir", EVAL_OUTPUT_DIR,
]
print("Iniciando comparação base vs. adapter com progresso...", flush=True)
log_path = os.path.join(WORKDIR, "outputs", "logs", f"{STAGE}_comparison.log")
os.makedirs(os.path.dirname(log_path), exist_ok=True)
process = subprocess.Popen(
    command,
    cwd=WORKDIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=0,
)
decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
assert process.stdout is not None
with open(log_path, "wb") as log_file:
    while True:
        chunk = os.read(process.stdout.fileno(), 4096)
        if not chunk:
            break
        log_file.write(chunk)
        log_file.flush()
        sys.stdout.write(decoder.decode(chunk))
        sys.stdout.flush()
    sys.stdout.write(decoder.decode(b"", final=True))
    sys.stdout.flush()
return_code = process.wait()
print(f"\nLog salvo em: {log_path}", flush=True)
if return_code:
    raise subprocess.CalledProcessError(return_code, command)

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

comparison_path = Path(WORKDIR) / EVAL_OUTPUT_DIR / "comparison.md"
display(Markdown(comparison_path.read_text(encoding="utf-8")))
print("Avalie A/B acima antes de abrir mapping.json.")
print("Planilha de notas:", Path(WORKDIR) / EVAL_OUTPUT_DIR / "ratings.json")

## Depois da avaliação

Somente após atribuir as notas, abra `outputs/evaluations/<stage>_smoke/mapping.json` para revelar qual resposta veio do modelo-base e qual veio do adapter. O próximo treino deve ser decidido a partir dessa comparação, não apenas pelo `eval_loss`.